In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.4 MB/s eta 0:00:00


In [ ]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from clearml import Task

In [ ]:
from tqdm import tqdm

In [ ]:
task = Task.init(
    project_name="cross-lingual-lm",
    task_name="swahili_adapter_roberta_low_resource",
    task_type=Task.TaskTypes.training
)

logger = task.get_logger()

set_seed(42)

ClearML Task: created new task id=e8a3be2eb51149208fdd39dd6e79ff7e


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-04-20 14:54:18,233 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/9bf855ce67034d8d9b44bf6ae4fbf457/experiments/e8a3be2eb51149208fdd39dd6e79ff7e/output/log


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize"

SAMPLE_SIZE = 10000

OUTPUT_DIR = "./swahili-adapter"


In [ ]:
task.connect({})

{}

In [ ]:
from datasets import load_dataset
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

# Remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Limit dataset size (low-resource simulation)
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

train_data = raw_train_data.select(range(min(SAMPLE_SIZE, len(raw_train_data))))
eval_data = eval_data.select(range(min(3000, len(eval_data))))

print(f"Train size: {len(train_data)}")
print(f"Eval size:  {len(eval_data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



README.md: 0.00B [00:00, ?B/s]

Swahili_Corpus_combined.txt:   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Train size: 10000
Eval size:  3000


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)

Loading weights:   0%|          | 0/202 [00:01<?, ?it/s]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 589,824 || all params: 91,331,293 || trainable%: 0.6458


In [ ]:
def group_texts(examples, block_size=128):
    # Собираем все тексты в одну кучу
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # Отбрасываем остаток, который меньше block_size
    total_length = (total_length // block_size) * block_size

    # Режем на равные куски по block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }

    # Для MLM нам нужны labels (коллатор их сделает сам, но можно подготовить)
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
tokenized_datasets_wide = train_data.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)
eval_tokenized = eval_data.map(
    lambda x: tokenizer(x["text"], truncation=True, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

# 2. Группируем в блоки по 256
lm_dataset_256 = tokenized_datasets_wide.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)

# Сделай то же самое для валидационного набора
eval_lm_dataset_256 = eval_tokenized.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
from transformers import TrainingArguments, Trainer

args_lora = TrainingArguments(
    output_dir="./results_lora",
    per_device_train_batch_size=16,
    learning_rate=2e-4,           # Для LoRA можно и нужно чуть выше
    num_train_epochs=30,          # Давай возьмем 30, чтобы увидеть динамику
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=200,             # Плавный разогрев для стабильности
    logging_steps=50,
    fp16=True,
    save_strategy="no",
    report_to="clearml"
)

trainer_lora = Trainer(
    model=model,
    args=args_lora,
    train_dataset=lm_dataset_256,
    data_collator=data_collator
)

trainer_lora.train()

2026-04-20 15:01:59,308 - clearml.Task - WARNING - Parameters must be of builtin type (Transformers/accelerator_config[AcceleratorConfig])


Step,Training Loss
50,6.030336
100,5.967241
150,5.956267
200,5.958019
250,5.935563
300,5.923510
350,5.929894
400,5.917878
450,5.897617
500,5.912751


TrainOutput(global_step=2430, training_loss=5.861920542775849, metrics={'train_runtime': 471.8949, 'train_samples_per_second': 82.391, 'train_steps_per_second': 5.149, 'total_flos': 5150467724820480.0, 'train_loss': 5.861920542775849, 'epoch': 30.0})

In [ ]:
trainer_lora.save_model(OUTPUT_DIR)

In [ ]:
from peft import PeftModel

base_model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
import math
import torch
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 8

In [ ]:
def eval_ppl(MODEL_PATH, dataset):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    base_model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    model.to(device)
    model.eval()

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    args = TrainingArguments(
        output_dir="./eval_en_tmp",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        eval_dataset=dataset,
        data_collator=collator
    )


    torch.manual_seed(42)
    metrics = trainer.evaluate()

    loss = metrics["eval_loss"]
    perplexity = math.exp(loss)

    print(f"Loss: {loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")

In [ ]:
eval_ppl(MODEL_PATH, eval_lm_dataset_256)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 5.7717
Perplexity: 321.09


In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)

examples = [
    f"Habari za {tokenizer.mask_token}?",
    f"Mimi ni {tokenizer.mask_token}.",
    f"Jina langu ni {tokenizer.mask_token}."
]

for ex in examples:
    print(f"\nPrompt: {ex}")
    for res in fill_mask(ex):
        print(f"  {res['score']:.4f} -> {res['token_str']}")


Prompt: Habari za <mask()?
  0.0118 ->  mwaka
  0.0097 ->  tanzania
  0.0076 ->  serikali
  0.0058 ->  na
  0.0052 ->  kwa

Prompt: Mimi ni <mask().
  0.0378 ->  
  0.0094 ->  kwa
  0.0089 ->  ya
  0.0089 ->  wa
  0.0076 ->  na

Prompt: Jina langu ni <mask().
  0.0525 ->  
  0.0065 ->  m
  0.0065 ->  kwa
  0.0060 ->  ya
  0.0057 ->  wa


In [ ]:
import shutil
import os

shutil.copytree(OUTPUT_DIR, '/content/drive/MyDrive/nlp_project/models/roberta_swahili_adapter', dirs_exist_ok=True)

'/content/drive/MyDrive/nlp_project/models/roberta_swahili_adapter'

# Evaluate on EN PPL

In [ ]:
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

MAX_LENGTH = 128
BATCH_SIZE = 8
EVAL_SAMPLES = 3000

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

dataset = dataset.filter(lambda x: x["text"] and len(x["text"].strip()) > 0)
dataset = dataset.select(range(min(EVAL_SAMPLES, len(dataset))))

dataset

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2891
})

In [ ]:
tokenized_dataset = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512, add_special_tokens=True),
    batched=True,
    remove_columns=["text"]
)

eval_en = tokenized_dataset.map(
    group_texts,
    batched=True,
    fn_kwargs={"block_size": 256}
)


Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

In [ ]:
eval_ppl(MODEL_PATH, eval_en)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 6.0098
Perplexity: 407.40


# Evaluate on Swahili PPL

In [ ]:
eval_ppl(MODEL_PATH, eval_lm_dataset_256)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loss: 5.7717
Perplexity: 321.09
